# Baseline FER: HOG + SVM trên Google Colab
### Ảnh xám thô, không dùng chuỗi chuẩn hóa ánh sáng đề xuất

- **Created**: 2026-11-25T17:17:20+07:00
- **Last Updated**: 2026-11-25T17:17:20+07:00
- **Author**: Nhóm dự án FER
- **Objective**: Huấn luyện baseline HOG với cùng dữ liệu và cấu hình SVM để so sánh với pipeline LBP.

---

## References & Consumed Artifacts

- **Source Code**: [Train CLI](../src/training/train.py), [HOG](../src/feature_extraction/hog_descriptor.py), [model](../src/emotion_recognition/svm_fer_classifier.py), [evaluation](../src/evaluation/evaluate.py).
- **Artifacts**: [Model local](../models/), [ảnh train](../data/raw/train/), [ảnh test giữ riêng](../data/raw/test/).
- **Governing Rule**: [Huấn luyện bằng script](../agents/rules/LOGGING_CHECKPOINT_RULES.md).
- **References**: [OpenCV HOG](https://docs.opencv.org/4.12.0/d5/d33/structcv_1_1HOGDescriptor.html), [Colab FAQ](https://research.google.com/colaboratory/faq.html).

Baseline cố định: ảnh 128×128, cell 16×16, block 32×32, stride 16×16, 9 bins, L2-Hys, tắt gamma của OpenCV HOG; tổng 1764 đặc trưng. SVM RBF: C=10, gamma=scale, class_weight=balanced, giống chính sách cấu hình model LBP. Không chọn tham số từ test. GPU không tự tăng tốc SVC; fit không có resume.


## 1. Upload gói baseline mới

Dùng gói `.tmp/fer2013_hog_colab.zip` mới, không dùng gói LBP cũ vì code cũ chưa có HOG. Nếu cần tạo lại, chạy `.venv\Scripts\python -m src.training.package_colab --output .tmp/fer2013_hog_colab_v2.zip`. Chỉ upload gói do bạn tạo. Thư mục Colab riêng giúp tránh dùng nhầm code đã giải nén từ notebook LBP.


In [ ]:
import json
import sys
import zipfile
from pathlib import Path
from google.colab import files

CONTENT_DIR = Path("/content")
PROJECT_DIR = CONTENT_DIR / "fer_hog_baseline"
manifest_path = PROJECT_DIR / "colab_manifest.json"
if manifest_path.is_file():
    print("Reusing the extracted project:", PROJECT_DIR)
else:
    if PROJECT_DIR.exists():
        raise RuntimeError("Project folder is incomplete. Use a fresh runtime; no files were deleted.")
    archive = CONTENT_DIR / "fer2013_hog_colab.zip"
    if not archive.is_file():
        print("Select fer2013_hog_colab.zip from your computer.")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Select exactly one ZIP bundle.")
        archive = Path(next(iter(uploaded)))
        if not archive.is_absolute():
            archive = CONTENT_DIR / archive
        del uploaded
    if not archive.is_file():
        raise FileNotFoundError(f"Upload did not create the selected file: {archive}")
    with zipfile.ZipFile(archive) as bundle:
        required = {"colab_manifest.json", "requirements-colab.txt", "src/training/train.py", "src/feature_extraction/hog_descriptor.py"}
        if not required.issubset(bundle.namelist()):
            raise ValueError("This is not the FER project bundle. Select fer2013_hog_colab.zip.")
        for member in bundle.infolist():
            (PROJECT_DIR / member.filename).resolve().relative_to(PROJECT_DIR.resolve())
        bundle.extractall(PROJECT_DIR)

manifest = json.loads(manifest_path.read_text())
if not (PROJECT_DIR / "src/training/train.py").is_file() or not (PROJECT_DIR / "data/raw/train").is_dir():
    raise RuntimeError("Extracted project is incomplete; use a fresh runtime.")
if not (PROJECT_DIR / "src/feature_extraction/hog_descriptor.py").is_file():
    raise RuntimeError("Old source bundle: upload the new HOG bundle in a fresh runtime.")
print("Colab kernel:", sys.version.split()[0])
print("Training Python:", ".".join(manifest["python_major_minor"]))
print("Training images:", manifest["training_images"])


## 2. Tạo môi trường train khớp máy local

Kernel Colab có thể là Python 3.13. Ô này dùng uv tải Python 3.12 và tạo môi trường riêng theo phiên bản trong gói, rồi cài đúng các thư viện đã chốt. Kernel vẫn giữ nguyên; script train và kiểm tra model chạy bằng `TRAIN_PYTHON`. Cần kết nối mạng để tải Python/thư viện. Nếu có lỗi cài đặt, dừng tại đây, không chạy ô train.


In [ ]:
import subprocess

training_version = ".".join(manifest["python_major_minor"])
TRAIN_ENV = CONTENT_DIR / ("fer_train_py" + training_version.replace(".", ""))
TRAIN_PYTHON = TRAIN_ENV / "bin/python"
if not TRAIN_PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
    subprocess.run([
        sys.executable, "-m", "uv", "venv", "--python", training_version,
        "--seed", str(TRAIN_ENV),
    ], check=True)

actual_version = subprocess.check_output([
    str(TRAIN_PYTHON), "-c",
    "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')",
], text=True).strip()
if actual_version != training_version:
    raise RuntimeError(f"Training environment is Python {actual_version}, expected {training_version}.")
subprocess.run([
    str(TRAIN_PYTHON), "-m", "pip", "install", "-r",
    str(PROJECT_DIR / "requirements-colab.txt"),
], check=True)
environment_json = subprocess.check_output([
    str(TRAIN_PYTHON), "-c",
    "import json, platform; from importlib.metadata import version; print(json.dumps({'python_version': platform.python_version(), 'dependencies': {name: version(name) for name in ['numpy', 'scikit-learn', 'scipy', 'joblib', 'Pillow', 'opencv-python-headless']}}))",
], text=True)
(PROJECT_DIR / "training_environment.json").write_text(environment_json)
print("Training interpreter:", TRAIN_PYTHON)
print(environment_json)


## 3. Kết nối Drive để giữ model hoàn tất

Chấp thuận kết nối Google Drive của bạn. Mỗi lần chạy tạo thư mục riêng dưới `MyDrive/FER_Lab`; không ghi đè lần chạy cũ. Nếu không gắn Drive được, dừng tại đây thay vì chỉ lưu model trong ổ tạm của Colab.


In [ ]:
import shutil
from datetime import datetime
from google.colab import drive

drive.mount("/content/drive")
RUN_DIR = Path("/content/drive/MyDrive/FER_Lab/HOG_Baseline") / datetime.now().strftime("%Y%m%d_%H%M%S_hog")
RUN_DIR.mkdir(parents=True, exist_ok=False)
MODEL_PATH = RUN_DIR / "hog_baseline.joblib"
shutil.copy2(PROJECT_DIR / "colab_manifest.json", RUN_DIR / "colab_manifest.json")
shutil.copy2(PROJECT_DIR / "training_environment.json", RUN_DIR / "training_environment.json")
print("Model will be saved to:", MODEL_PATH)


## 4. Chạy baseline HOG + SVM

Gọi cùng CLI nhưng thêm `--feature-type hog`. Chỉ dùng ảnh train, giữ nguyên test để đánh giá sau. Trích đặc trưng báo tiến độ mỗi 500 ảnh; SVM in log của LIBSVM, không phải phần trăm hoàn thành hoặc checkpoint. Không chạy lại ô khi tiến trình cũ còn đang fit.


In [ ]:
if "TRAIN_ENV" not in globals():
    raise RuntimeError("Run step 2 successfully before training.")
TRAIN_PYTHON = Path(TRAIN_ENV) / "bin/python"
if not TRAIN_PYTHON.is_file():
    raise RuntimeError("Training Python is missing. Rerun step 2.")

command = [
    str(TRAIN_PYTHON), "-u", "-m", "src.training.train",
    "--data-dir", str(PROJECT_DIR / "data/raw/train"),
    "--cropped-faces", "--feature-type", "hog", "--output", str(MODEL_PATH),
]
with (RUN_DIR / "train.log").open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed. Read {RUN_DIR / 'train.log'} before using any artifact.")
print("Training completed:", MODEL_PATH)


## 5. Kiểm tra model và tải về

Chỉ chạy ô này sau khi ô train hoàn tất thành công. Kiểm tra nạp model và nhãn trước khi tải. Model cũng nằm trên Drive để có thể tải lại sau khi Colab hết phiên.


In [ ]:
if return_code != 0 or not MODEL_PATH.is_file():
    raise RuntimeError("No successfully trained model to download.")
subprocess.run([
    str(TRAIN_PYTHON), "-c",
    "import sys; from pathlib import Path; from src.emotion_recognition.svm_fer_classifier import load_classifier; model = load_classifier(Path(sys.argv[1]), feature_type='hog'); print('Verified labels:', model.classes_)",
    str(MODEL_PATH),
], cwd=PROJECT_DIR, check=True)
files.download(str(MODEL_PATH))


## 6. Đánh giá baseline trên máy

Đặt model tải về vào `models/hog_baseline.joblib`, giữ nguyên model LBP `models/fer.joblib`. Chạy trên đúng cùng test và chế độ cắt mặt:

```powershell
.venv\Scripts\python -m src.evaluation.evaluate --model models/hog_baseline.joblib --feature-type hog --data-dir data/raw/test --cropped-faces
```

So sánh Accuracy, Macro-F1 và từng lớp với báo cáo LBP. HOG+SVM là baseline của toàn pipeline; để quy đóng góp riêng cho Gamma/DoG vẫn cần ablation trên cùng đặc trưng LBP. Notebook chưa chạy E1–E4 hoặc khảo sát tham số.
